<a href="https://colab.research.google.com/github/thofaa/inflation_YoY_prediction/blob/main/Transformation_Code_Inflation_YoY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Min-Max Normalization for M1 and M2

First, I'll load the `M2M1_Data_2017_2026.json` file into a pandas DataFrame.

In [2]:
from pathlib import Path
import pandas as pd


# Load the JSON data
df = pd.read_json('./data/M2M1_Norm_2017_2026.json')

# Display the original DataFrame and its info
print("Original DataFrame head:")
display(df.head())
print("\nDataFrame Info:")
df.info()

Original DataFrame head:


,year,month,M1,M2,M1_normalized,M2_normalized
0,2017,1,1191499.69,4936881.99,0.0000,0.0000
1,2017,2,1196036.61,4942919.76,0.0020,0.0011
2,2017,3,1215856.68,5017643.55,0.0108,0.0147
3,2017,4,1245927.39,5033780.29,0.0242,0.0176
4,2017,5,1275892.50,5125383.79,0.0376,0.0343



DataFrame Info:
<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   year           120 non-null    int64  
 1   month          120 non-null    int64  
 2   M1             114 non-null    float64
 3   M2             114 non-null    float64
 4   M1_normalized  114 non-null    float64
 5   M2_normalized  114 non-null    float64
dtypes: float64(4), int64(2)
memory usage: 5.8 KB


Now, it will be applied Min-Max Normalization to the 'M1' and 'M2' columns separately using `sklearn.preprocessing.MinMaxScaler`. This ensures that each column is scaled independently based on its own minimum and maximum values.

In [3]:
from sklearn.preprocessing import MinMaxScaler

# Initialize MinMaxScaler for M1
mins_scaler_m1 = MinMaxScaler()

# Reshape 'M1' column to a 2D array as required by MinMaxScaler and round
df['M1_normalized'] = mins_scaler_m1.fit_transform(df[['M1']]).round(4)

# Initialize MinMaxScaler for M2
mins_scaler_m2 = MinMaxScaler()

# Reshape 'M2' column to a 2D array and round
df['M2_normalized'] = mins_scaler_m2.fit_transform(df[['M2']]).round(4)

# Display the DataFrame with normalized 'M1' and 'M2' values
print("DataFrame with Normalized M1 and M2 columns:")
display(df[['M1', 'M1_normalized', 'M2', 'M2_normalized']].head())

DataFrame with Normalized M1 and M2 columns:


,M1,M1_normalized,M2,M2_normalized
0,1191499.69,0.0000,4936881.99,0.0000
1,1196036.61,0.0020,4942919.76,0.0011
2,1215856.68,0.0108,5017643.55,0.0147
3,1245927.39,0.0242,5033780.29,0.0176
4,1275892.50,0.0376,5125383.79,0.0343


Now, I will save the DataFrame with the normalized values to a new JSON file named `M2M1_Norm_2017_2026.json`.

In [ ]:
# Save the DataFrame to a new JSON file
df.to_json(base_dir/'data'/'M2M1_Norm_2017_2026.json', orient='records', indent=4)

print("Normalized data saved to /home/makoo/Documents/codinghell/inflation_YoY_prediction/data/M2M1_Norm_2017_2026.json")

Normalized data saved to M2M1_Norm_2017_2026.json


In [8]:
import json


with open('./data/BI_Rate_2017_2026.json', 'r') as file:
    bi_rate = file.read()
    df_bi_rate = pd.DataFrame(json.loads(bi_rate))
    display(df_bi_rate)

,NO,Tanggal,BI Rate
0,1,19 Agustus 2026,5.75 %
1,2,22 Juli 2026,5.75 %
2,3,18 Juni 2026,5.75 %
3,4,9 Juni 2026,5.50 %
4,5,20 Mei 2026,5.25 %
...,...,...,...
113,114,18 Mei 2017,4.75 %
114,115,20 April 2017,4.75 %
115,116,16 Maret 2017,4.75 %
116,117,16 Februari 2017,4.75 %


In [16]:
for element in df_bi_rate["BI Rate"]:
    if element.strip().endswith('%'):
        new = float(str(element.strip())[0:4].strip())
    df_bi_rate.replace(element, new, inplace=True)
display(df_bi_rate)

,NO,Tanggal,BI Rate
0,1,19 Agustus 2026,5.75
1,2,22 Juli 2026,5.75
2,3,18 Juni 2026,5.75
3,4.0,9 Juni 2026,5.5
4,5.0,20 Mei 2026,5.25
...,...,...,...
113,114,18 Mei 2017,4.75
114,115,20 April 2017,4.75
115,116,16 Maret 2017,4.75
116,117,16 Februari 2017,4.75


In [18]:
with open('./data/Inflation_YoY_2017_2026.json', 'r') as file:
    inflasi = file.read()
    df_inflasi = pd.DataFrame(json.loads(inflasi))
    display(df_inflasi)

,No,Periode,Inflasi
0,1,Juli 2026,2.88 %
1,2,Juni 2026,3.34 %
2,3,Mei 2026,3.08 %
3,4,April 2026,2.42 %
4,5,Maret 2026,3.48 %
...,...,...,...
110,111,Mei 2017,4.33 %
111,112,April 2017,4.17 %
112,113,Maret 2017,3.61 %
113,114,Februari 2017,3.83 %


In [ ]:
import pandas as pd

# Convert inflation strings like '3 %' or '3.5%' into numeric values
inflasi_values = (
    df_inflasi["Inflasi"].astype(str).str.extract(r"([-+]?\d*\.?\d+)")[0]
)
df_inflasi["Inflasi"] = pd.to_numeric(inflasi_values, errors="coerce")
display(df_inflasi)


ValueError: could not convert string to float: '3 %'